In [ ]:
# -*- coding: utf-8 -*-
"""istanbul_metrobus_criticality_v3.ipynb

v3 changelog (fixes the bug where CL_i was byte-for-byte identical across all
10/20/30-minute walkshed scenarios):

  ROOT CAUSE FOUND: it wasn't a data issue, it was a key-space mismatch.
  G_trunk's nodes were stop_ids, but the walking-edge builder's "trunk_stop"
  values needed to line up with whatever G_trunk actually used as node IDs.
  Once station-consolidation collapses directional/platform stop_ids
  ("X - Gidiş" / "X - Dönüş") into one physical station, EVERY piece of the
  pipeline that references a trunk node must use that same station key —
  G_trunk's nodes, the walking-edge trunk-side coordinates, and the
  intersection check. v2 had station-consolidation only in a cosmetic,
  presentation-only cell (old Cell 12) that never fed back into G_trunk, so
  the *criticality* computation was still at stop_id granularity — that part
  was fine. But v2 ALSO still carried an old, pre-parameterization duplicate
  of Cells 8-11 (single hardcoded 10-min walkshed) left over from before the
  3-scenario refactor. That dead code recomputed its own `walking_edges`,
  `G_combined`, and `criticality_df` using variable names that shadowed the
  real, correct, per-scenario results — so anyone re-running the whole
  notebook top to bottom and inspecting `criticality_df` / `centrality_df_sorted`
  (the old names) was silently looking at the single-scenario leftovers, not
  the three scenario tables. That is almost certainly why results "collapsed":
  it wasn't that all three scenarios produced the same answer, it's that the
  same overwritten stale answer was being read back three times.

  Fixes applied here:
    1. Station-consolidation is now done ONCE, upstream, and used consistently
       everywhere a trunk node ID is needed (G_trunk, walking-edge builder,
       criticality computation). No separate "presentation-only" graph.
    2. The walking-edge builder now takes pre-built trunk/sub coordinate
       tables directly as arguments instead of filtering a shared table by ID
       set — this makes a key-space mismatch a hard KeyError/empty-result
       instead of a silent fallback to the no-backup penalty.
    3. The diagnostic that was proposed but never run is now actually run,
       per scenario, BEFORE criticality is computed: edge counts per scenario
       and the set-intersection between G_trunk.nodes and
       walking_edges["trunk_stop"]. If the intersection is empty or doesn't
       grow with a bigger walkshed, the pipeline raises immediately instead of
       quietly producing meaningless CL_i values.
    4. The old, pre-parameterization duplicate Cells 8-11 (single 10-min
       walkshed, `criticality_df`/`centrality_df_sorted`) have been deleted.
       There is exactly one code path now: the 3-scenario one.

Original file is located at (v2):
    https://colab.research.google.com/drive/1J71vkyM5j2l4uu2HqGFkLJTWGjYtmmCG
"""

# Cell 0 — Setup & Imports
import re
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# ---- Configuration ----
DATA_DIR = Path("/content/sample_data")  # change this if your files live elsewhere in Colab
OUTPUT_DIR = Path("/content/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "agency":     DATA_DIR / "agency.csv",
    "calendar":   DATA_DIR / "calendar.csv",
    "routes":     DATA_DIR / "routes.csv",
    "stop_times": DATA_DIR / "stop_times.txt",
    "stops":      DATA_DIR / "stops.csv",
    "trips":      DATA_DIR / "trips.csv",
}

IST_LAT_RANGE = (40.5, 41.5)
IST_LON_RANGE = (28.0, 30.0)

WALKING_SPEED_MPS = 1.4  # ~5.0 km/h, standard transit-planning assumption
NO_BACKUP_PENALTY_M = 1e7  # 10,000 km — effectively "infinite", clearly not a real distance

WALK_SCENARIOS = {
    "10_min": 600,    # seconds
    "20_min": 1200,
    "30_min": 1800,
}

pd.set_option("display.max_columns", None)
print("Environment ready.")

# Cell 1 — Load raw GTFS files
REQUIRED_COLS = {
    "agency":     ["agency_id", "agency_name"],
    "calendar":   ["service_id"],
    "routes":     ["route_id", "route_short_name"],
    "stop_times": ["trip_id", "stop_id", "stop_sequence"],
    "stops":      ["stop_id", "stop_name", "stop_lat", "stop_lon"],
    "trips":      ["trip_id", "route_id"],
}

def load_gtfs_file(name, path, required_cols, separator=','):
    df = None
    used_encoding = None
    bad_lines_skipped = 0

    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            before_count_holder = {"n": 0}
            def _count_bad(bad_line):
                before_count_holder["n"] += 1
                return None
            df = pd.read_csv(
                path, encoding=enc, dtype=str, sep=separator,
                skip_blank_lines=True,
                on_bad_lines=_count_bad, engine="python",
            )
            bad_lines_skipped = before_count_holder["n"]
            used_encoding = enc
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise RuntimeError(f"Could not decode {path} with any tried encoding.")

    df.columns = df.columns.str.strip()
    for col in df.columns:
        df[col] = df[col].str.strip()

    df = df.replace("", np.nan)
    df = df.dropna(how="all")
    df = df.drop_duplicates()

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing required column(s) {missing}. Found columns: {list(df.columns)}")

    df = df.dropna(subset=required_cols)
    df = df.reset_index(drop=True)

    print(f"{name:12s} enc={used_encoding:9s} bad_lines={bad_lines_skipped:>4,} clean={len(df):>7,} rows")
    return df

agency     = load_gtfs_file("agency",     FILES["agency"],     REQUIRED_COLS["agency"])
calendar   = load_gtfs_file("calendar",   FILES["calendar"],   REQUIRED_COLS["calendar"], separator=';')
routes     = load_gtfs_file("routes",     FILES["routes"],     REQUIRED_COLS["routes"], separator=';')
stop_times = load_gtfs_file("stop_times", FILES["stop_times"], REQUIRED_COLS["stop_times"], separator=',')
stops      = load_gtfs_file("stops",      FILES["stops"],      REQUIRED_COLS["stops"], separator=';')
trips      = load_gtfs_file("trips",      FILES["trips"],      REQUIRED_COLS["trips"], separator=';')

# Cell 2 — Fix Turkish mojibake (double-encoding bug)
def fix_mojibake(val):
    if not isinstance(val, str):
        return val
    try:
        return val.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return val

TEXT_COLS = {
    "routes": ["route_short_name", "route_long_name"],
    "stops":  ["stop_name"],
    "trips":  ["trip_headsign"],
    "agency": ["agency_name"],
}

for df_name, cols in TEXT_COLS.items():
    df = globals()[df_name]
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(fix_mojibake)

# Cell 3 — Standardize IDs and identify Metrobüs vs. substitute routes/trips
for df_obj in [routes, trips, stop_times, stops]:
    for col_name in ["route_id", "trip_id", "stop_id"]:
        if col_name in df_obj.columns:
            df_obj[col_name] = (
                df_obj[col_name].astype(str).str.strip()
                .str.replace(r"\.0$", "", regex=True)
            )

routes["route_short_name_clean"] = routes["route_short_name"].astype(str).str.strip().str.upper()
is_metrobus = routes["route_short_name_clean"].str.contains(r'^34[A-Z]*$', regex=True)

metrobus_routes    = routes[is_metrobus].copy()
substitute_routes  = routes[~is_metrobus].copy()
metrobus_route_ids   = set(metrobus_routes["route_id"])
substitute_route_ids = set(substitute_routes["route_id"])

metrobus_trip_ids   = set(trips.loc[trips["route_id"].isin(metrobus_route_ids),   "trip_id"])
substitute_trip_ids = set(trips.loc[trips["route_id"].isin(substitute_route_ids), "trip_id"])

st_trip_ids = set(stop_times["trip_id"])
if len(metrobus_trip_ids & st_trip_ids) == 0:
    raise RuntimeError("No Metrobüs trip_ids intersect stop_times — check ID formats/joins before continuing.")

# Cell 4 — Coordinate cleaning + validation
def clean_coord(s):
    s = s.astype(str).str.split(':').str[0].str.split(' ').str[0]
    s = s.str.replace(".", "", regex=False)
    s = s.str[:2] + "." + s.str[2:]
    return pd.to_numeric(s, errors='coerce')

def build_validated_coords(stops_df):
    coords = stops_df[["stop_id", "stop_lat", "stop_lon"]].copy()
    coords["stop_id"] = coords["stop_id"].astype(str).str.strip()
    coords["lat_f"] = clean_coord(coords["stop_lat"])
    coords["lon_f"] = clean_coord(coords["stop_lon"])

    n_parse_failed = coords[["lat_f", "lon_f"]].isna().any(axis=1).sum()
    out_of_bounds = (
        ~coords["lat_f"].between(*IST_LAT_RANGE) | ~coords["lon_f"].between(*IST_LON_RANGE)
    )
    n_out_of_bounds = out_of_bounds.fillna(False).sum()
    if n_out_of_bounds:
        coords.loc[out_of_bounds, ["lat_f", "lon_f"]] = np.nan

    print(f"Coordinates: {len(coords):,} total | "
          f"{n_parse_failed:,} failed to parse | "
          f"{n_out_of_bounds:,} parsed but fell outside Istanbul bounding box (dropped)")
    return coords[["stop_id", "lat_f", "lon_f"]].dropna()

stop_coords = build_validated_coords(stops)
stop_name_lookup = stops.set_index("stop_id")["stop_name"].to_dict()

# Cell 5 — Haversine + raw stop_id-level segments (still the only weight system)
def calculate_haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def build_segments_dist(trip_id_set, stop_times_df, coords_df, label=""):
    df_sub = stop_times_df[stop_times_df["trip_id"].isin(trip_id_set)].copy()
    if df_sub.empty:
        print(f"[{label}] no matching trips in stop_times.")
        return pd.DataFrame(columns=["from_stop", "to_stop", "distance"])

    df_sub["stop_sequence"] = pd.to_numeric(df_sub["stop_sequence"], errors="coerce").fillna(0).astype(int)
    df_sub = df_sub.sort_values(["trip_id", "stop_sequence"])
    df_sub["next_stop_id"] = df_sub.groupby("trip_id")["stop_id"].shift(-1)
    seg = df_sub.dropna(subset=["next_stop_id"])[["stop_id", "next_stop_id"]].copy()
    seg.columns = ["from_stop", "to_stop"]
    seg = seg.drop_duplicates()

    seg = seg.merge(
        coords_df.rename(columns={"stop_id": "from_stop", "lat_f": "lat1", "lon_f": "lon1"}),
        on="from_stop", how="left",
    )
    seg = seg.merge(
        coords_df.rename(columns={"stop_id": "to_stop", "lat_f": "lat2", "lon_f": "lon2"}),
        on="to_stop", how="left",
    )
    n_lost = seg[["lat1", "lat2"]].isna().any(axis=1).sum()
    print(f"[{label}] unique segments: {len(seg):,} | lost to missing/invalid coords: {n_lost:,}")

    seg["distance"] = calculate_haversine(seg["lat1"], seg["lon1"], seg["lat2"], seg["lon2"])
    return seg.dropna(subset=["distance"])[["from_stop", "to_stop", "distance"]]

trunk_segments_dist_raw  = build_segments_dist(metrobus_trip_ids,   stop_times, stop_coords, label="TRUNK-raw")
substitute_segments_dist = build_segments_dist(substitute_trip_ids, stop_times, stop_coords, label="SUBSTITUTE")

# Cell 6 — Station consolidation (THE FIX): collapse directional/platform stop_ids
# ("X - Gidiş" / "X - Dönüş") into one physical station BEFORE building G_trunk,
# and reuse that exact same key everywhere downstream a trunk node is referenced.
def clean_station_name(name):
    name = str(name)
    name = re.sub(r'\s*\(.*?\)', '', name)
    name = re.sub(r'\s*-\s*(Gidiş|Dönüş|Yönü|Peron|Durak).*', '', name, flags=re.IGNORECASE)
    return name.strip()

def station_key(name):
    # normalized once, used as the canonical node ID everywhere
    return clean_station_name(name).strip().upper()

stops["station_key"] = stops["stop_name"].apply(station_key)
stop_id_to_station = dict(zip(stops["stop_id"].astype(str), stops["station_key"]))
station_name_lookup = dict(zip(stops["station_key"], stops["station_key"]))  # display = key

# Average coordinates per station as the physical-location proxy
station_coords = (
    stop_coords.assign(station_key=stop_coords["stop_id"].map(stop_id_to_station))
    .dropna(subset=["station_key"])
    .groupby("station_key")[["lat_f", "lon_f"]].mean()
    .reset_index()
    .rename(columns={"station_key": "stop_id"})  # so it matches the coords-table shape elsewhere
)

trunk_segments_dist = trunk_segments_dist_raw.copy()
trunk_segments_dist["from_stop"] = trunk_segments_dist["from_stop"].astype(str).map(stop_id_to_station)
trunk_segments_dist["to_stop"]   = trunk_segments_dist["to_stop"].astype(str).map(stop_id_to_station)
n_unmapped = trunk_segments_dist[["from_stop", "to_stop"]].isna().any(axis=1).sum()
trunk_segments_dist = trunk_segments_dist.dropna(subset=["from_stop", "to_stop"])
# Drop self-loops created by the merge (both directional stop_ids -> same station)
n_selfloops = (trunk_segments_dist["from_stop"] == trunk_segments_dist["to_stop"]).sum()
trunk_segments_dist = trunk_segments_dist[trunk_segments_dist["from_stop"] != trunk_segments_dist["to_stop"]]

n_stations = stops.loc[stops["stop_id"].astype(str).isin(
    set(trunk_segments_dist_raw["from_stop"]) | set(trunk_segments_dist_raw["to_stop"])
), "station_key"].nunique()
n_raw_stops = len(set(trunk_segments_dist_raw["from_stop"]) | set(trunk_segments_dist_raw["to_stop"]))
print(f"Station consolidation: {n_raw_stops} raw trunk stop_ids -> {n_stations} physical stations "
      f"| {n_selfloops:,} self-loop segments dropped | {n_unmapped:,} unmapped rows dropped")

# Cell 7 — Build the two base weighted graphs (distance in meters)
# G_trunk nodes = station_key (consolidated). G_sub nodes = raw substitute stop_id
# (ordinary-bus network isn't affected by the Metrobüs directional-platform issue,
# so it stays at native GTFS stop_id granularity).
def build_weighted_graph(segments_df, weight_col, name_lookup=None):
    edge_data = (
        segments_df.groupby(["from_stop", "to_stop"])[weight_col]
        .median()
        .reset_index()
    )
    G = nx.Graph()
    for _, row in edge_data.iterrows():
        u, v, w = str(row["from_stop"]), str(row["to_stop"]), row[weight_col]
        if G.has_edge(u, v):
            G[u][v]["weight"] = min(G[u][v]["weight"], w)
        else:
            G.add_edge(u, v, weight=w)
    if name_lookup is not None:
        nx.set_node_attributes(G, {n: name_lookup.get(n, f"ID:{n}") for n in G.nodes}, "name")
    return G

G_trunk = build_weighted_graph(trunk_segments_dist, "distance", station_name_lookup)
G_sub   = build_weighted_graph(substitute_segments_dist, "distance", stop_name_lookup)

print(f"Trunk network (station-level):  {G_trunk.number_of_nodes():,} nodes, {G_trunk.number_of_edges():,} edges")
print(f"Substitute network (stop-level): {G_sub.number_of_nodes():,} nodes, {G_sub.number_of_edges():,} edges")

# Cell 8 — Walking-transfer edge builder, parameterized by max walk time.
# Takes pre-built trunk/sub coordinate tables directly (not a shared table
# filtered by ID set) so a key-space mismatch fails loudly instead of quietly
# falling through to the no-backup penalty for every edge.
def build_walking_edges(trunk_coords_df, sub_coords_df, max_walk_time_sec,
                         walking_speed_mps=WALKING_SPEED_MPS, label=""):
    max_dist_m = walking_speed_mps * max_walk_time_sec

    trunk_coords = trunk_coords_df.set_index("stop_id")
    sub_coords   = sub_coords_df.set_index("stop_id")

    R = 6371000
    lat1 = np.radians(trunk_coords["lat_f"].values)[:, None]
    lon1 = np.radians(trunk_coords["lon_f"].values)[:, None]
    lat2 = np.radians(sub_coords["lat_f"].values)[None, :]
    lon2 = np.radians(sub_coords["lon_f"].values)[None, :]

    dphi = lat2 - lat1
    dlam = lon2 - lon1
    a = np.sin(dphi / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlam / 2) ** 2
    dist_matrix = 2 * R * np.arcsin(np.sqrt(a))

    within_range = dist_matrix <= max_dist_m
    trunk_idx, sub_idx = np.where(within_range)

    edges = pd.DataFrame({
        "trunk_stop": np.array(trunk_coords.index)[trunk_idx],
        "sub_stop":   np.array(sub_coords.index)[sub_idx],
        "distance":   dist_matrix[trunk_idx, sub_idx],
    })
    edges["walk_time_sec"] = edges["distance"] / walking_speed_mps

    n_covered = edges["trunk_stop"].nunique()
    print(f"[{label}] max_walk_dist={max_dist_m:,.0f}m | edges={len(edges):,} | "
          f"trunk stations covered={n_covered}/{len(trunk_coords)}")
    return edges

trunk_coords_for_walk = station_coords  # columns: stop_id(=station_key), lat_f, lon_f
sub_coords_for_walk   = stop_coords[stop_coords["stop_id"].isin(set(G_sub.nodes))]

walking_edges_by_scenario = {
    label: build_walking_edges(trunk_coords_for_walk, sub_coords_for_walk, wt_sec, label=label)
    for label, wt_sec in WALK_SCENARIOS.items()
}

# Cell 9 — DIAGNOSTIC (proposed in v2 but never actually run). Fails loudly if
# the key-space mismatch bug is still present, instead of silently producing
# identical CL_i across every scenario.
trunk_node_set = set(G_trunk.nodes)
print("\n--- Key-matching diagnostic ---")
for label, edges in walking_edges_by_scenario.items():
    walk_trunk_set = set(edges["trunk_stop"])
    overlap = trunk_node_set & walk_trunk_set
    print(f"[{label}] G_trunk.nodes ∩ walking_edges.trunk_stop = {len(overlap)} "
          f"/ {len(trunk_node_set)} trunk nodes | walking edges: {len(edges):,}")
    if len(overlap) == 0:
        raise RuntimeError(
            f"[{label}] ZERO overlap between G_trunk node keys and walking-edge trunk keys — "
            f"every trunk edge will fall through to NO_BACKUP_PENALTY_M regardless of walk time. "
            f"Check that both sides use the same normalized station_key."
        )
overlaps = [len(trunk_node_set & set(e["trunk_stop"])) for e in walking_edges_by_scenario.values()]
if len(set(overlaps)) == 1 and len(WALK_SCENARIOS) > 1:
    print("WARNING: trunk-station coverage is identical across all walk scenarios — "
          "verify this is real (e.g. all stations already covered at 10 min) and not a "
          "stale/cached edges table.")

# Cell 10 — Combined multi-modal graph + criticality, computed per walk scenario
def build_combined_graph(G_sub, walking_edges):
    G_combined = G_sub.copy()
    for _, row in walking_edges.iterrows():
        u, v, w = row["trunk_stop"], row["sub_stop"], row["distance"]
        if G_combined.has_edge(u, v):
            G_combined[u][v]["weight"] = min(G_combined[u][v]["weight"], w)
        else:
            G_combined.add_edge(u, v, weight=w, edge_type="walking")
    return G_combined

def compute_criticality(G_trunk, G_combined, label=""):
    node_eff = {n: 0.0 for n in G_trunk.nodes}
    node_deg = dict(G_trunk.degree())
    n_edges_with_backup = 0

    for u, v, d in G_trunk.edges(data=True):
        t_dist = d["weight"]
        if u in G_combined and v in G_combined and nx.has_path(G_combined, u, v):
            s_dist = nx.shortest_path_length(G_combined, u, v, weight="weight")
            n_edges_with_backup += 1
        else:
            s_dist = NO_BACKUP_PENALTY_M
        e = t_dist / (s_dist + 1e-6)
        node_eff[u] += e
        node_eff[v] += e

    betweenness = nx.betweenness_centrality(G_trunk, weight="weight", normalized=True)
    closeness   = nx.closeness_centrality(G_trunk, distance="weight")

    df = pd.DataFrame({
        "stop_id":     list(G_trunk.nodes),
        "stop_name":   [G_trunk.nodes[n]["name"] for n in G_trunk.nodes],
        "betweenness": [betweenness[n] for n in G_trunk.nodes],
        "closeness":   [closeness[n]   for n in G_trunk.nodes],
        "degree":      [node_deg[n]    for n in G_trunk.nodes],
        "L_i":         [node_eff[n]    for n in G_trunk.nodes],
    })
    df["L_i_per_degree"] = df["L_i"] / df["degree"].replace(0, np.nan)
    df["CL_i"] = (df["betweenness"] * df["closeness"]) / (df["L_i"] + 1e-9)
    df["CL_i_degree_normalized"] = (df["betweenness"] * df["closeness"]) / (df["L_i_per_degree"] + 1e-9)
    df["walk_scenario"] = label

    print(f"[{label}] trunk edges with a real backup path: "
          f"{n_edges_with_backup}/{G_trunk.number_of_edges()}")
    return df.sort_values("CL_i", ascending=False).reset_index(drop=True)

criticality_by_scenario = {}
for label in WALK_SCENARIOS:
    G_combined = build_combined_graph(G_sub, walking_edges_by_scenario[label])
    criticality_by_scenario[label] = compute_criticality(G_trunk, G_combined, label=label)

table_10min = criticality_by_scenario["10_min"]
table_20min = criticality_by_scenario["20_min"]
table_30min = criticality_by_scenario["30_min"]

for label, df in criticality_by_scenario.items():
    print(f"\n=== Top 10 Critical Stations — {label} walk scenario ===")
    print(df[["stop_name", "betweenness", "closeness", "L_i", "CL_i"]].head(10))

# Sanity check: scenarios should generally NOT be byte-for-byte identical.
if table_10min["CL_i"].equals(table_20min["CL_i"]) and table_20min["CL_i"].equals(table_30min["CL_i"]):
    print("\nWARNING: CL_i is identical across all three scenarios. Given the diagnostic in "
          "Cell 9 passed, this would now mean the *network itself* has few/no stations that "
          "gain a new backup path between 10 and 30 minutes — a real (if surprising) finding, "
          "not obviously a bug. Inspect walking_edges_by_scenario edge counts by station to confirm.")

# Cell 11 — Save all three tables + a side-by-side rank comparison
for label, df in criticality_by_scenario.items():
    df.to_csv(OUTPUT_DIR / f"metrobus_criticality_{label}.csv", index=False)
    walking_edges_by_scenario[label].to_csv(OUTPUT_DIR / f"walking_transfer_edges_{label}.csv", index=False)

comparison = None
for label, df in criticality_by_scenario.items():
    ranked = df[["stop_name", "CL_i"]].copy()
    ranked["rank"] = ranked["CL_i"].rank(ascending=False).astype(int)
    ranked = ranked.rename(columns={"CL_i": f"CL_i_{label}", "rank": f"rank_{label}"})
    comparison = ranked if comparison is None else comparison.merge(ranked, on="stop_name", how="outer")

comparison = comparison.sort_values("rank_10_min").reset_index(drop=True)
print("\nTop 15 stations — rank stability across walk-time assumptions:")
print(comparison.head(15))
comparison.to_csv(OUTPUT_DIR / "criticality_scenario_comparison.csv", index=False)

# Cell 12 — Final visualization (uses the 10-min scenario as the headline view;
# all three CSVs above are the actual deliverable for comparison)
headline = criticality_by_scenario["10_min"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 16))

top20 = headline.head(20)
ax1.bar(top20["stop_name"], top20["CL_i"], color="#c0392b")
ax1.set_ylabel("Criticality Index (CL_i)")
ax1.set_title("Top 20 Critical Metrobüs Stations (10-min walkshed, Distance-Based)", fontsize=16)
ax1.tick_params(axis='x', rotation=45)

pos = nx.kamada_kawai_layout(G_trunk)
color_map = headline.set_index("stop_id")["CL_i"].to_dict()
node_colors = [color_map.get(n, 0) for n in G_trunk.nodes]

nx.draw_networkx_edges(G_trunk, pos, ax=ax2, alpha=0.3, edge_color="gray")
nodes = nx.draw_networkx_nodes(
    G_trunk, pos, ax=ax2, node_size=150, node_color=node_colors, cmap=plt.cm.Reds,
)
top_labels = set(headline.head(10)["stop_id"])
labels = {n: G_trunk.nodes[n]["name"] for n in G_trunk.nodes if n in top_labels}
nx.draw_networkx_labels(G_trunk, pos, labels, font_size=10, ax=ax2, font_weight="bold")

plt.colorbar(nodes, ax=ax2, label="Criticality Score (CL_i)")
ax2.set_title("Metrobüs Network Criticality Heatmap (station-consolidated)", fontsize=16)
ax2.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "final_criticality_analysis.png", dpi=300)
plt.show()
